In [1]:
## pydantic
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000168F4861B10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000168F4861A20>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This is the year the movie was released.")
    director:str=Field(description="This is the director of the movie")
    rating:float=Field(description="movie rating out of 10")
    

In [3]:
model_with_structure = model.with_structured_output(Movie)

In [4]:
model_with_structure.invoke("Provide details of inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [11]:
#Message output alongside parsed structure

model_with_structure_output = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure_output.invoke("Provide details about the movie RaOne.")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Provide details about the movie RaOne." We need to provide details. Could use function Movie? The function expects director, rating, title, year. We can call function to output details. Probably we need to call function with appropriate info: director: Anubhav Sinha? Wait, Ra.One is a 2011 Indian sci-fi action film directed by Anubhav Sinha, starring Shah Rukh Khan. Rating? Could give maybe 5.5/10? But we can provide rating out of 10. Provide approximate rating e.g., 4.5? Let\'s assume 5.0. Use function.\n\nWe\'ll call functions.Movie with details.', 'tool_calls': [{'id': 'fc_316155ab-820d-40f1-9aa5-7e0cf2dea665', 'function': {'arguments': '{"director":"Anubhav Sinha","rating":5,"title":"Ra.One","year":2011}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 195, 'prompt_tokens': 162, 'total_tokens': 357, 'completion_time': 0.413634997, 'completion_toke

In [12]:
#Nested Structure
class actor(BaseModel):
    name:str
    role:str

class MovieDetail(BaseModel):
    title: str
    year: int
    cast: list[actor]
    genre: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

In [13]:
model_with_strc = model.with_structured_output(MovieDetail)

In [14]:
model_with_strc.invoke("tell me about movie lagaan.")

MovieDetail(title='Lagaan: Once Upon a Time in India', year=2001, cast=[actor(name='Aamir Khan', role='Bhuvan'), actor(name='Gracy Singh', role='Gauri'), actor(name='Rachel Shelley', role='Elizabeth Russell'), actor(name='Paul Blackthorne', role='Captain Andrew Russell'), actor(name='Kulbhushan Kharbanda', role='British Officer')], genre=['Sports', 'Drama', 'Historical'], budget=None)